# Practical worksheet: Evaluation and Exploration of Generative ML Spaces
## STUDENT VERSION

This practical follows the theory lecture on navigation and evaluation in generative spaces.
The notebook uses a pretrained diffusion model, so there is no training stage.

## Learning focus

- Implement `lerp` and `slerp` for navigation in a generative noise space
- Visualize the transition from point A to point B
- Define an evaluation protocol and implement the FID core
- Compare `real vs real`, `real vs generated`, and `real vs random noise`
- First explore MNIST, then explore an RGB `diffusers` model

Why this sequence:
- MNIST is lighter and easier to inspect while learning the workflow
- After that, the notebook moves to RGB generation so you can transfer the same ideas beyond grayscale digits


## Notebook setup

Run the next cell once to install the dependencies from `requirements.txt`.

In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import DDPMPipeline
from scipy import linalg
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import Inception_V3_Weights, inception_v3
from torchvision.utils import make_grid


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device('mps')
    return torch.device('cpu')


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


set_seed(7)
device = get_device()
print(f'Device: {device}')

## Helper functions

The notebook keeps data loading, visualization, evaluation helpers, and local checkpoint model definitions inline.
This makes the notebook easier to run in Colab or VS Code without extra files.


In [ ]:
def load_mnist_subset(limit=128, train=False, data_root='data'):
    transform = transforms.ToTensor()
    dataset = datasets.MNIST(data_root, train=train, download=True, transform=transform)
    if limit is None or limit >= len(dataset):
        subset = dataset
    else:
        subset = Subset(dataset, list(range(limit)))

    loader = DataLoader(subset, batch_size=min(64, len(subset)), shuffle=False, num_workers=0)
    images, labels = [], []
    for xb, yb in loader:
        images.append(xb)
        labels.append(yb)
    return torch.cat(images, dim=0), torch.cat(labels, dim=0)


def find_oxford_pet_annotation_file(data_root='data', split='test'):
    data_root = Path(data_root)
    for path in data_root.rglob(f'{split}.txt'):
        if path.parent.name == 'annotations':
            return path
    raise FileNotFoundError(f'Could not find Oxford-IIIT Pet annotation file for split={split!r} under {data_root}.')


def load_oxford_pet_cat_subset(limit=128, split='test', image_size=256, data_root='data'):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])
    dataset = datasets.OxfordIIITPet(
        data_root,
        split=split,
        target_types='category',
        download=True,
        transform=transform,
    )

    annotation_path = find_oxford_pet_annotation_file(data_root=data_root, split=split)
    cat_indices = []
    with annotation_path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            species = int(parts[2])
            if species == 1:  # Oxford-IIIT Pet metadata: 1 = cat, 2 = dog
                cat_indices.append(idx)

    if not cat_indices:
        raise ValueError('No cat samples were found in the Oxford-IIIT Pet split.')

    if limit is not None:
        cat_indices = cat_indices[: min(limit, len(cat_indices))]

    subset = Subset(dataset, cat_indices)
    loader = DataLoader(subset, batch_size=min(32, len(subset)), shuffle=False, num_workers=0)
    images, labels = [], []
    for xb, yb in loader:
        images.append(xb)
        labels.append(yb)
    return torch.cat(images, dim=0), torch.cat(labels, dim=0)


def show_image_grid(images, title=None, nrow=8, figsize=(8, 8), cmap='gray'):
    images = images.detach().cpu().float().clamp(0, 1)
    grid = make_grid(images, nrow=nrow)
    np_grid = grid.permute(1, 2, 0).numpy()

    plt.figure(figsize=figsize)
    if np_grid.shape[-1] == 1:
        plt.imshow(np_grid[..., 0], cmap=cmap)
    else:
        plt.imshow(np_grid)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_transition_strip(images, t_values, title):
    images = images.detach().cpu().float().clamp(0, 1)
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(2.0 * n, 2.5))
    if n == 1:
        axes = [axes]
    for ax, img, t in zip(axes, images, t_values):
        if img.shape[0] == 1:
            ax.imshow(img[0], cmap='gray')
        else:
            ax.imshow(img.permute(1, 2, 0))
        ax.set_title(f't={float(t):.2f}')
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_interpolation_comparison(lerp_images, slerp_images, t_values, title):
    lerp_images = lerp_images.detach().cpu().float().clamp(0, 1)
    slerp_images = slerp_images.detach().cpu().float().clamp(0, 1)
    n = len(t_values)
    fig, axes = plt.subplots(2, n, figsize=(2.0 * n, 4.0))
    for row_idx, (row_axes, images, label) in enumerate(zip(axes, [lerp_images, slerp_images], ['lerp', 'slerp'])):
        for col_idx, (ax, img, t) in enumerate(zip(row_axes, images, t_values)):
            if img.shape[0] == 1:
                ax.imshow(img[0], cmap='gray')
            else:
                ax.imshow(img.permute(1, 2, 0))
            if row_idx == 0:
                ax.set_title(f't={float(t):.2f}')
            if col_idx == 0:
                ax.set_ylabel(label)
            ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
def load_pretrained_ddpm(model_id='1aurent/ddpm-mnist', cache_dir='hf-cache', device=None):
    if device is None:
        device = get_device()
    cache_path = Path(cache_dir)
    cache_path.mkdir(parents=True, exist_ok=True)

    pipeline = DDPMPipeline.from_pretrained(model_id, cache_dir=str(cache_path))
    pipeline.set_progress_bar_config(disable=True)
    pipeline = pipeline.to(device)
    pipeline.unet.eval()
    return pipeline


def ddpm_noise_shape(pipeline, batch_size):
    sample_size = pipeline.unet.config.sample_size
    if isinstance(sample_size, int):
        height = width = sample_size
    else:
        height, width = sample_size
    return (batch_size, pipeline.unet.config.in_channels, height, width)


def make_noise_batch(pipeline, batch_size, seed, device=None):
    if device is None:
        device = get_device()
    generator = torch.Generator(device='cpu').manual_seed(seed)
    noise = torch.randn(ddpm_noise_shape(pipeline, batch_size), generator=generator)
    return noise.to(device)


@torch.no_grad()
def sample_ddpm_from_noise(pipeline, noise, num_inference_steps=50):
    pipeline.scheduler.set_timesteps(num_inference_steps, device=noise.device)
    sample = noise.to(device=noise.device, dtype=pipeline.unet.dtype)

    for t in pipeline.scheduler.timesteps:
        noise_pred = pipeline.unet(sample, t).sample
        sample = pipeline.scheduler.step(noise_pred, t, sample).prev_sample

    return (sample / 2 + 0.5).clamp(0, 1).float()


@torch.no_grad()
def sample_ddpm_from_noises(pipeline, noises, num_inference_steps=50, batch_size=16):
    outputs = []
    for start in range(0, len(noises), batch_size):
        batch = noises[start:start + batch_size]
        outputs.append(sample_ddpm_from_noise(pipeline, batch, num_inference_steps=num_inference_steps))
    return torch.cat(outputs, dim=0)


@torch.no_grad()
def sample_random_ddpm(pipeline, n_samples, seed=123, num_inference_steps=50):
    model_device = next(pipeline.unet.parameters()).device
    noises = make_noise_batch(pipeline, batch_size=n_samples, seed=seed, device=model_device)
    return sample_ddpm_from_noises(pipeline, noises, num_inference_steps=num_inference_steps, batch_size=min(16, n_samples))

In [ ]:
def build_inception_feature_extractor(device=None):
    if device is None:
        device = get_device()
    model = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False)
    model.fc = nn.Identity()
    model.eval()
    return model.to(device)


def prepare_images_for_inception(images, device):
    if images.ndim == 3:
        images = images.unsqueeze(0)
    images = images.to(device=device, dtype=torch.float32)
    if images.shape[1] == 1:
        images = images.repeat(1, 3, 1, 1)
    images = images.clamp(0, 1)
    images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)

    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
    return (images - mean) / std


@torch.no_grad()
def extract_inception_features(images, model, batch_size=32, device=None):
    if device is None:
        device = next(model.parameters()).device
    features = []
    for start in range(0, len(images), batch_size):
        batch = prepare_images_for_inception(images[start:start + batch_size], device=device)
        feats = model(batch)
        if isinstance(feats, tuple):
            feats = feats[0]
        features.append(feats.detach().cpu())
    return torch.cat(features, dim=0)


def feature_statistics(features):
    features = np.asarray(features, dtype=np.float64)
    mu = features.mean(axis=0)
    sigma = np.cov(features, rowvar=False)
    return mu, sigma


def plot_2d_paths(lerp_fn, slerp_fn, steps=9):
    a = torch.tensor([1.0, 0.0])
    b = torch.tensor([0.0, 1.0])
    t_values = torch.linspace(0.0, 1.0, steps=steps)
    lerp_path = torch.stack([lerp_fn(a, b, float(t)) for t in t_values], dim=0)
    slerp_path = torch.stack([slerp_fn(a, b, float(t)) for t in t_values], dim=0)

    theta = np.linspace(0.0, 2.0 * np.pi, 300)
    plt.figure(figsize=(5, 5))
    plt.plot(np.cos(theta), np.sin(theta), linestyle='--', color='lightgray', label='unit circle')
    plt.plot(lerp_path[:, 0], lerp_path[:, 1], marker='o', label='lerp')
    plt.plot(slerp_path[:, 0], slerp_path[:, 1], marker='o', label='slerp')
    plt.scatter([a[0], b[0]], [a[1], b[1]], color='black', zorder=3)
    plt.text(a[0] + 0.03, a[1] + 0.03, 'A')
    plt.text(b[0] + 0.03, b[1] + 0.03, 'B')
    plt.axis('equal')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('2D geometry of lerp vs slerp')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Notebook-local model definitions and checkpoint helpers

This notebook includes the local model definitions inline so it can run in Colab or in editors such as VS Code without relying on an external `.py` file.
The same definitions are also mirrored in `generative_spaces_lab.py` for local reuse.


In [ ]:
import math
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn


def resolve_device(device: Optional[torch.device | str] = None) -> torch.device:
    if isinstance(device, torch.device):
        return device
    if isinstance(device, str):
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    return torch.device("cpu")


def denorm(images: torch.Tensor) -> torch.Tensor:
    return (images + 1.0) / 2.0


def make_noise_batch_from_shape(
    sample_shape: tuple[int, ...],
    batch_size: int,
    seed: int,
    device: Optional[torch.device | str] = None,
) -> torch.Tensor:
    generator = torch.Generator(device="cpu").manual_seed(seed)
    noise = torch.randn((batch_size, *sample_shape), generator=generator)
    return noise.to(resolve_device(device))


class GaussianDiffusion:
    def __init__(self, num_timesteps=1000, beta_start=0.0001, beta_end=0.02, device="cpu"):
        self.num_timesteps = num_timesteps
        self.device = resolve_device(device)
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=self.device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.cat(
            [torch.tensor([1.0], device=self.device), self.alphas_cumprod[:-1]]
        )
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        sqrt_alpha_prod = self._get_index(self.sqrt_alphas_cumprod, t, x_0.shape)
        sqrt_one_minus_alpha_prod = self._get_index(self.sqrt_one_minus_alphas_cumprod, t, x_0.shape)
        return sqrt_alpha_prod * x_0 + sqrt_one_minus_alpha_prod * noise

    @torch.no_grad()
    def p_sample(self, model, x, t, t_index):
        betas_t = self._get_index(self.betas, t, x.shape)
        sqrt_one_minus_alpha_cumprod_t = self._get_index(self.sqrt_one_minus_alphas_cumprod, t, x.shape)
        sqrt_recip_alphas_t = 1.0 / torch.sqrt(self._get_index(self.alphas, t, x.shape))
        predicted_noise = model(x, t)
        model_mean = sqrt_recip_alphas_t * (
            x - betas_t * predicted_noise / sqrt_one_minus_alpha_cumprod_t
        )

        if t_index == 0:
            return model_mean

        posterior_variance_t = self._get_index(self.posterior_variance, t, x.shape)
        noise = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_variance_t) * noise

    def _get_index(self, tensor, t, x_shape):
        out = tensor.gather(-1, t)
        return out.view(t.shape[0], *((1,) * (len(x_shape) - 1)))


class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        half_dim = self.dim // 2
        scale = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=x.device) * -scale)
        emb = x[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)


class ResnetBlock(nn.Module):
    def __init__(self, dim, time_emb_dim, out_dim=None):
        super().__init__()
        self.out_dim = out_dim or dim
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, self.out_dim))
        self.conv1 = nn.Conv2d(dim, self.out_dim, 3, padding=1)
        self.conv2 = nn.Conv2d(self.out_dim, self.out_dim, 3, padding=1)
        self.norm1 = nn.GroupNorm(4, dim)
        self.norm2 = nn.GroupNorm(4, self.out_dim)
        self.act = nn.SiLU()
        self.shortcut = nn.Conv2d(dim, self.out_dim, 1) if dim != self.out_dim else nn.Identity()

    def forward(self, x, time_emb):
        h = self.norm1(x)
        h = self.act(h)
        h = self.conv1(h)
        h = h + self.mlp(time_emb)[:, :, None, None]
        h = self.norm2(h)
        h = self.act(h)
        h = self.conv2(h)
        return self.shortcut(x) + h


class LatentDenoiseNetwork(nn.Module):
    def __init__(self, latent_channels=4, model_channels=64, num_res_blocks=3):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(model_channels),
            nn.Linear(model_channels, model_channels * 4),
            nn.SiLU(),
            nn.Linear(model_channels * 4, model_channels * 4),
        )
        self.init_conv = nn.Conv2d(latent_channels, model_channels, 3, padding=1)
        self.res_blocks = nn.ModuleList(
            [ResnetBlock(model_channels, model_channels * 4) for _ in range(num_res_blocks)]
        )
        self.out_conv = nn.Conv2d(model_channels, latent_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = self.init_conv(x)
        for block in self.res_blocks:
            h = block(h, t_emb)
        return self.out_conv(h)


class PixelUNet(nn.Module):
    def __init__(self, in_channels=1, model_channels=64):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(model_channels),
            nn.Linear(model_channels, model_channels * 4),
            nn.SiLU(),
            nn.Linear(model_channels * 4, model_channels * 4),
        )
        time_dim = model_channels * 4
        self.init_conv = nn.Conv2d(in_channels, model_channels, 3, padding=1)
        self.down1_res = ResnetBlock(model_channels, time_dim)
        self.down1_pool = nn.Conv2d(model_channels, model_channels, 3, stride=2, padding=1)
        self.down2_res = ResnetBlock(model_channels, time_dim, out_dim=model_channels * 2)
        self.down2_pool = nn.Conv2d(model_channels * 2, model_channels * 2, 3, stride=2, padding=1)
        self.mid_res1 = ResnetBlock(model_channels * 2, time_dim)
        self.mid_res2 = ResnetBlock(model_channels * 2, time_dim)
        self.up2_conv = nn.ConvTranspose2d(model_channels * 2, model_channels, 4, stride=2, padding=1)
        self.up2_res = ResnetBlock(model_channels * 3, time_dim, out_dim=model_channels)
        self.up1_conv = nn.ConvTranspose2d(model_channels, model_channels, 4, stride=2, padding=1)
        self.up1_res = ResnetBlock(model_channels * 2, time_dim, out_dim=model_channels)
        self.out_conv = nn.Conv2d(model_channels, in_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h_init = self.init_conv(x)
        h1 = self.down1_res(h_init, t_emb)
        h1_pool = self.down1_pool(h1)
        h2 = self.down2_res(h1_pool, t_emb)
        h2_pool = self.down2_pool(h2)
        h_mid = self.mid_res1(h2_pool, t_emb)
        h_mid = self.mid_res2(h_mid, t_emb)
        h_up2 = self.up2_conv(h_mid)
        h_up2 = torch.cat([h_up2, h2], dim=1)
        h_up2 = self.up2_res(h_up2, t_emb)
        h_up1 = self.up1_conv(h_up2)
        h_up1 = torch.cat([h_up1, h1], dim=1)
        h_up1 = self.up1_res(h_up1, t_emb)
        return self.out_conv(h_up1)


class Encoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.mu = nn.Conv2d(64, latent_channels, kernel_size=1)
        self.logvar = nn.Conv2d(64, latent_channels, kernel_size=1)

    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)


class Decoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.initial_conv = nn.Conv2d(latent_channels, 64, kernel_size=1)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(self.initial_conv(z))


class VAE(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.encoder = Encoder(latent_channels)
        self.decoder = Decoder(latent_channels)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


@dataclass
class PixelCheckpointBundle:
    checkpoint_path: Path
    diffusion: GaussianDiffusion
    model: PixelUNet
    sample_shape: tuple[int, int, int] = (1, 28, 28)


@dataclass
class LatentCheckpointBundle:
    checkpoint_path: Path
    diffusion: GaussianDiffusion
    model: LatentDenoiseNetwork
    vae: VAE
    sample_shape: tuple[int, int, int] = (4, 4, 4)


def load_checkpoint_dict(
    checkpoint_path: Path | str,
    device: Optional[torch.device | str] = None,
) -> dict:
    checkpoint_path = Path(checkpoint_path)
    device = resolve_device(device)

    try:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location=device)
    except pickle.UnpicklingError as exc:
        prefix = checkpoint_path.read_bytes()[:16]
        if prefix.lstrip().startswith(b"<"):
            raise ValueError(
                f"{checkpoint_path} does not look like a PyTorch checkpoint. "
                "It looks like HTML, which usually means the download URL points to a web page "
                "or a Google Drive folder instead of a direct file."
            ) from exc
        raise ValueError(
            f"Could not load {checkpoint_path} as a PyTorch checkpoint. "
            "If this file came from the web, verify that it is a direct .pt file download."
        ) from exc

    if not isinstance(checkpoint, dict):
        raise ValueError(
            f"Expected a checkpoint dictionary in {checkpoint_path}, got {type(checkpoint).__name__}."
        )
    return checkpoint


def load_local_pixel_ddpm(
    checkpoint_path: Path | str = Path("checkpoints") / "ddpm_pixel_model.pt",
    device: Optional[torch.device | str] = None,
) -> PixelCheckpointBundle:
    checkpoint_path = Path(checkpoint_path)
    device = resolve_device(device)
    checkpoint = load_checkpoint_dict(checkpoint_path, device=device)

    diffusion = GaussianDiffusion(
        num_timesteps=checkpoint["num_timesteps"],
        device=device,
    )
    model = PixelUNet(
        in_channels=checkpoint.get("in_channels", 1),
        model_channels=checkpoint.get("model_channels", 64),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return PixelCheckpointBundle(
        checkpoint_path=checkpoint_path,
        diffusion=diffusion,
        model=model,
    )


def load_local_latent_ddpm(
    checkpoint_path: Path | str = Path("checkpoints") / "ddpm_latent_model.pt",
    device: Optional[torch.device | str] = None,
) -> LatentCheckpointBundle:
    checkpoint_path = Path(checkpoint_path)
    device = resolve_device(device)
    checkpoint = load_checkpoint_dict(checkpoint_path, device=device)
    latent_channels = checkpoint.get("latent_channels", 4)

    diffusion = GaussianDiffusion(
        num_timesteps=checkpoint["num_timesteps"],
        device=device,
    )
    vae = VAE(latent_channels=latent_channels).to(device)
    vae.load_state_dict(checkpoint["vae_state_dict"])
    vae.eval()
    for param in vae.parameters():
        param.requires_grad = False

    model = LatentDenoiseNetwork(latent_channels=latent_channels).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return LatentCheckpointBundle(
        checkpoint_path=checkpoint_path,
        diffusion=diffusion,
        model=model,
        vae=vae,
        sample_shape=(latent_channels, 4, 4),
    )


@torch.no_grad()
def sample_local_diffusion_from_noise(
    model: nn.Module,
    diffusion: GaussianDiffusion,
    noise: torch.Tensor,
) -> torch.Tensor:
    x = noise.to(diffusion.device)
    model.eval()
    for step in reversed(range(diffusion.num_timesteps)):
        t = torch.full((x.shape[0],), step, device=diffusion.device, dtype=torch.long)
        x = diffusion.p_sample(model, x, t, step)
    return x


@torch.no_grad()
def sample_local_pixel_ddpm_from_noises(
    bundle: PixelCheckpointBundle,
    noises: torch.Tensor,
    batch_size: int = 8,
) -> torch.Tensor:
    outputs = []
    for start in range(0, len(noises), batch_size):
        batch = noises[start:start + batch_size]
        samples = sample_local_diffusion_from_noise(bundle.model, bundle.diffusion, batch)
        outputs.append(denorm(samples).clamp(0, 1).float())
    return torch.cat(outputs, dim=0)


@torch.no_grad()
def sample_local_latent_ddpm_from_noises(
    bundle: LatentCheckpointBundle,
    noises: torch.Tensor,
    batch_size: int = 8,
) -> torch.Tensor:
    outputs = []
    for start in range(0, len(noises), batch_size):
        batch = noises[start:start + batch_size]
        latents = sample_local_diffusion_from_noise(bundle.model, bundle.diffusion, batch)
        decoded = bundle.vae.decoder(latents)
        outputs.append(denorm(decoded).clamp(0, 1).float())
    return torch.cat(outputs, dim=0)


def sample_local_pixel_ddpm(
    bundle: PixelCheckpointBundle,
    n_samples: int,
    seed: int = 123,
    batch_size: int = 8,
) -> torch.Tensor:
    noises = make_noise_batch_from_shape(
        bundle.sample_shape,
        batch_size=n_samples,
        seed=seed,
        device=bundle.diffusion.device,
    )
    return sample_local_pixel_ddpm_from_noises(bundle, noises, batch_size=batch_size)


def sample_local_latent_ddpm(
    bundle: LatentCheckpointBundle,
    n_samples: int,
    seed: int = 123,
    batch_size: int = 8,
) -> torch.Tensor:
    noises = make_noise_batch_from_shape(
        bundle.sample_shape,
        batch_size=n_samples,
        seed=seed,
        device=bundle.diffusion.device,
    )
    return sample_local_latent_ddpm_from_noises(bundle, noises, batch_size=batch_size)


## Real reference data

We use a small MNIST subset as the reference distribution.
This keeps the practical fast and makes it easier to see whether the generated outputs remain on-manifold.

In [ ]:
real_images, real_labels = load_mnist_subset(limit=128, train=False, data_root='data')
print('Real tensor shape:', tuple(real_images.shape))
show_image_grid(real_images[:64], title='Real MNIST samples', nrow=8, figsize=(8, 8))

## Pretrained diffusion model from `diffusers`

This keeps the fast MNIST DDPM baseline loaded from Hugging Face.
The first run may download the weights to local cache.


In [ ]:
pipeline = load_pretrained_ddpm(model_id='1aurent/ddpm-mnist', cache_dir='hf-cache', device=device)
print('Loaded DDPM model:', pipeline.__class__.__name__)
print('UNet sample size:', pipeline.unet.config.sample_size)

## Checkpoint download from URL or Google Drive

Colab can load checkpoints from the web, but the link must resolve to a real file.

Use one of these:
- a direct `.pt` file URL
- a Google Drive file share link

Do not use a Google Drive folder link or a generic web page URL, because those usually download HTML instead of the checkpoint file.


In [ ]:
import subprocess
import sys
from pathlib import Path
import requests


def validate_checkpoint_file(destination, min_size_bytes=1024):
    destination = Path(destination)
    if not destination.exists():
        raise FileNotFoundError(destination)

    size_bytes = destination.stat().st_size
    if size_bytes < min_size_bytes:
        raise ValueError(
            f"Downloaded file is too small to be a checkpoint: {destination} ({size_bytes} bytes)."
        )

    prefix = destination.read_bytes()[:512].lstrip()
    if prefix.startswith(b"<") or prefix.lower().startswith(b"<!doctype html"):
        raise ValueError(
            f"{destination} looks like an HTML page, not a .pt checkpoint. "
            "Use a direct file URL or a Google Drive file share link, not a folder link."
        )

    return destination


def download_file(url, destination, chunk_size=1024 * 1024):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if 'drive.google.com/drive/folders/' in url:
        raise ValueError(
            'Google Drive folder links are not valid checkpoint downloads here. '
            'Use a direct file link or a Google Drive file share link.'
        )

    if 'drive.google.com' in url:
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
            import gdown
        gdown.download(url=url, output=str(destination), quiet=False, fuzzy=True)
        return validate_checkpoint_file(destination)

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(destination, 'wb') as handle:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    handle.write(chunk)
    return validate_checkpoint_file(destination)


def maybe_download_checkpoint(url, destination):
    destination = Path(destination)
    if destination.exists():
        try:
            print('Using existing checkpoint:', destination.resolve())
            return validate_checkpoint_file(destination)
        except ValueError as exc:
            if not url:
                raise ValueError(
                    f"Existing file for {destination.name} is invalid. Delete it or provide a valid URL."
                ) from exc
            print('Existing file is invalid, re-downloading:', destination.resolve())
            destination.unlink()
    if not url:
        print('No URL configured for:', destination.name)
        return destination
    print('Downloading', destination.name, 'from:', url)
    return download_file(url, destination)


checkpoints_dir = Path('checkpoints')
checkpoints_dir.mkdir(parents=True, exist_ok=True)

pixel_checkpoint_url = 'https://drive.google.com/file/d/1uir50YwfQDMfD8iDMLYSEQW-eNWQTp_S/view?usp=sharing'
latent_checkpoint_url = 'https://drive.google.com/file/d/1tY7bi80NuSzaz-h_pHvVXPs8YjKQ_AD0/view?usp=sharing'

maybe_download_checkpoint(pixel_checkpoint_url, checkpoints_dir / 'ddpm_pixel_model.pt')
maybe_download_checkpoint(latent_checkpoint_url, checkpoints_dir / 'ddpm_latent_model.pt')


## Models loaded from local checkpoints

This block keeps section 6 standalone.
It loads the local pixel DDPM and latent DDPM from the `checkpoints` folder beside the notebook.
If the files are missing, run the download cell above first.


In [ ]:
pixel_ckpt_path = Path('checkpoints') / 'ddpm_pixel_model.pt'
latent_ckpt_path = Path('checkpoints') / 'ddpm_latent_model.pt'

local_pixel = None
local_latent = None

print('Pixel checkpoint:', pixel_ckpt_path.resolve())
print('Latent checkpoint:', latent_ckpt_path.resolve())

if pixel_ckpt_path.exists():
    local_pixel = load_local_pixel_ddpm(pixel_ckpt_path, device=device)
    print('Loaded local pixel DDPM from:', local_pixel.checkpoint_path.resolve())
else:
    print('Pixel DDPM checkpoint not found.')

if latent_ckpt_path.exists():
    local_latent = load_local_latent_ddpm(latent_ckpt_path, device=device)
    print('Loaded local latent DDPM from:', local_latent.checkpoint_path.resolve())
else:
    print('Latent DDPM checkpoint not found.')

if local_pixel is not None:
    local_pixel_preview = sample_local_pixel_ddpm(local_pixel, n_samples=8, seed=101, batch_size=4)
    show_image_grid(local_pixel_preview, title='Local pixel DDPM samples', nrow=4, figsize=(4, 2.5))

if local_latent is not None:
    local_latent_preview = sample_local_latent_ddpm(local_latent, n_samples=8, seed=202, batch_size=4)
    show_image_grid(local_latent_preview, title='Local latent DDPM samples', nrow=4, figsize=(4, 2.5))


## Exercise 1: implement `lerp`

Goal:
- Move linearly from point $A$ to point $B$.

Formula:

$$
z(t) = (1 - t) z_0 + t z_1
$$

Pseudo-code:
1. Multiply $z_0$ by $(1 - t)$.
2. Multiply $z_1$ by $t$.
3. Add both terms.

Use the `# TODO START` and `# TODO END` markers below.


In [ ]:
def lerp(z0, z1, t):
    # TODO START
    # 1) Weight z0 by (1 - t)
    # 2) Weight z1 by t
    # 3) Return the sum
    raise NotImplementedError('Implement linear interpolation.')
    # TODO END

## Exercise 2: implement `slerp`

Goal:
- Move from point $A$ to point $B$ along the sphere instead of the straight chord.

Useful equations:

$$
u_0 = \frac{z_0}{\lVert z_0 \rVert}$$

$$
u_1 = \frac{z_1}{\lVert z_1 \rVert}$$

$$\omega = \arccos(\langle u_0, u_1 \rangle)$$

$$
\operatorname{slerp}(z_0, z_1, t) = \frac{\sin((1-t)\omega)}{\sin(\omega)} z_0 + \frac{\sin(t\omega)}{\sin(\omega)} z_1
$$

Algorithm sketch:
1. Flatten both tensors so the dot product is easy to compute.
2. Normalize them.
3. Compute the angle $\omega$.
4. If $\sin(\omega)$ is very small, fall back to `lerp`.
5. Otherwise apply the spherical formula.


In [ ]:
def slerp(z0, z1, t, eps=1e-7):
    # TODO START
    # 1) Flatten z0 and z1
    # 2) Normalize both vectors
    # 3) Compute dot product and omega = arccos(dot)
    # 4) If the angle is too small, return lerp(z0, z1, t)
    # 5) Otherwise return the spherical interpolation formula
    raise NotImplementedError('Implement spherical interpolation.')
    # TODO END

## Compare `lerp` and `slerp` through generated images

Use the decoded image sequences below as the main comparison, instead of focusing on vector trajectories.


In [ ]:
print('The comparison below focuses on generated images rather than vector trajectories.')


## Exercise 3: configure the interpolation experiment

Goal:
- Compare `lerp` and `slerp` using generated images, not vector plots.

What to configure:
- `interpolation_steps`: start with `10`
- `hf_num_inference_steps`: keep a fixed denoising budget for fair comparison

Why this matters:
- Too few interpolation steps can hide differences between the paths.
- Too many steps can make the strip harder to read in class.
- The interpolation protocol should be stated explicitly before comparing results.


In [ ]:
def build_interpolation_paths(z0, z1, t_values):
    lerp_path = torch.stack([lerp(z0, z1, float(t)) for t in t_values], dim=0)
    slerp_path = torch.stack([slerp(z0, z1, float(t)) for t in t_values], dim=0)
    return lerp_path, slerp_path


# TODO START
# Set the interpolation protocol.
# Start with 10 interpolation steps and keep the denoising budget fixed.
interpolation_steps = 10
hf_num_inference_steps = 50
# TODO END

t_values = torch.linspace(0.0, 1.0, steps=interpolation_steps, device=device)

hf_zA = make_noise_batch(pipeline, batch_size=1, seed=11, device=device)[0]
hf_zB = make_noise_batch(pipeline, batch_size=1, seed=29, device=device)[0]
hf_lerp_path, hf_slerp_path = build_interpolation_paths(hf_zA, hf_zB, t_values)

pixel_lerp_path = pixel_slerp_path = None
if local_pixel is not None:
    pixel_zA = make_noise_batch_from_shape(local_pixel.sample_shape, batch_size=1, seed=11, device=device)[0]
    pixel_zB = make_noise_batch_from_shape(local_pixel.sample_shape, batch_size=1, seed=29, device=device)[0]
    pixel_lerp_path, pixel_slerp_path = build_interpolation_paths(pixel_zA, pixel_zB, t_values)

latent_lerp_path = latent_slerp_path = None
if local_latent is not None:
    latent_zA = make_noise_batch_from_shape(local_latent.sample_shape, batch_size=1, seed=11, device=device)[0]
    latent_zB = make_noise_batch_from_shape(local_latent.sample_shape, batch_size=1, seed=29, device=device)[0]
    latent_lerp_path, latent_slerp_path = build_interpolation_paths(latent_zA, latent_zB, t_values)

print('Interpolation steps:', interpolation_steps)


In [ ]:
hf_lerp_images = sample_ddpm_from_noises(
    pipeline,
    hf_lerp_path,
    num_inference_steps=hf_num_inference_steps,
    batch_size=len(t_values),
)
hf_slerp_images = sample_ddpm_from_noises(
    pipeline,
    hf_slerp_path,
    num_inference_steps=hf_num_inference_steps,
    batch_size=len(t_values),
)
show_interpolation_comparison(hf_lerp_images, hf_slerp_images, t_values.cpu(), title='Diffusers MNIST DDPM: lerp vs slerp')

if pixel_lerp_path is not None:
    pixel_lerp_images = sample_local_pixel_ddpm_from_noises(local_pixel, pixel_lerp_path, batch_size=len(t_values))
    pixel_slerp_images = sample_local_pixel_ddpm_from_noises(local_pixel, pixel_slerp_path, batch_size=len(t_values))
    show_interpolation_comparison(pixel_lerp_images, pixel_slerp_images, t_values.cpu(), title='Local pixel DDPM: lerp vs slerp')

if latent_lerp_path is not None:
    latent_lerp_images = sample_local_latent_ddpm_from_noises(local_latent, latent_lerp_path, batch_size=len(t_values))
    latent_slerp_images = sample_local_latent_ddpm_from_noises(local_latent, latent_slerp_path, batch_size=len(t_values))
    show_interpolation_comparison(latent_lerp_images, latent_slerp_images, t_values.cpu(), title='Local latent DDPM: lerp vs slerp')


## Exercise 4: define the evaluation methodology

Goal:
- Write down the evaluation protocol before computing the score.

Checklist:
1. Choose the number of real reference images.
2. Choose the number of generated images.
3. Fix the feature-extraction batch size.
4. Fix the seeds used for generation and for the noise baseline.
5. Keep the comparison pairs explicit: $\text{real} \leftrightarrow \text{real}$, $\text{real} \leftrightarrow \text{generated}$, and $\text{real} \leftrightarrow \text{noise}$.

This exercise is about methodology first. The score is only meaningful after the protocol is fixed.


In [ ]:
# TODO START
# Adjust the protocol values below and keep them explicit.
evaluation_config = {
    'reference_count': 128,
    'generated_count': 128,
    'feature_batch_size': 32,
    'subset_size': 32,
    'num_subsets': 20,
    'generation_seed': 123,
    'noise_seed': 321,
    'num_inference_steps': 50,
}
# TODO END

if evaluation_config['reference_count'] % 2 != 0:
    raise ValueError('reference_count must be even so it can be split into two reference sets.')

feature_extractor = build_inception_feature_extractor(device=device)
real_eval_images = real_images[:evaluation_config['reference_count']]
generated_images = sample_random_ddpm(
    pipeline,
    n_samples=evaluation_config['generated_count'],
    seed=evaluation_config['generation_seed'],
    num_inference_steps=evaluation_config['num_inference_steps'],
)
noise_generator = torch.Generator(device='cpu').manual_seed(evaluation_config['noise_seed'])
noise_images = torch.rand(real_eval_images.shape, generator=noise_generator)

print('Evaluation protocol:', evaluation_config)
show_image_grid(real_eval_images[:64], title='Real evaluation reference samples', nrow=8, figsize=(8, 8))
show_image_grid(generated_images[:64], title='Generated MNIST samples', nrow=8, figsize=(8, 8))
show_image_grid(noise_images[:64], title='Random noise baseline', nrow=8, figsize=(8, 8))


## Exercise 5: implement only the FID core

Goal:
- Implement the compact Fréchet distance formula once the protocol and features are already prepared.

The Fréchet Inception Distance compares two Gaussian feature distributions with means $\mu_r, \mu_g$ and covariances $\Sigma_r, \Sigma_g$:

$$
\operatorname{FID} = \lVert \mu_r - \mu_g \rVert_2^2 + \operatorname{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)
$$

The emphasis here is not to rebuild an entire evaluation package.
The emphasis is to understand what the score means under a fixed protocol.


In [ ]:
def frechet_distance(mu_r, sigma_r, mu_g, sigma_g, eps=1e-6):
    # TODO START
    # 1) Convert inputs to float64 numpy arrays.
    # 2) Compute diff = mu_r - mu_g.
    # 3) Add eps * I for numerical stability.
    # 4) Compute the matrix square root of (Sigma_r + eps I)(Sigma_g + eps I).
    # 5) If the result is complex, keep only the real part.
    # 6) Return the final FID scalar as a Python float.
    raise NotImplementedError('Implement the FID core formula.')
    # TODO END


## Provided KID helper

`KID` is provided below so the notebook can spend more time on evaluation methodology and experimental design.
Use it as a secondary diagnostic, not as the main implementation task.

What is $\operatorname{MMD}$?
- $\operatorname{MMD}$ stands for Maximum Mean Discrepancy.
- It compares two distributions by mapping samples into a kernel feature space and measuring the distance between their mean embeddings.
- If two sample sets come from similar distributions, the $\operatorname{MMD}$ value should be small. If they come from different distributions, the value tends to be larger.
- `KID` is built from this idea: it estimates an $\operatorname{MMD}$ using a polynomial kernel on pretrained image features.


In [ ]:
def polynomial_kernel(x, y):
    d = x.shape[1]
    return ((x @ y.T) / d + 1.0) ** 3


def kid_score(features_real, features_fake, subset_size=64, num_subsets=10, seed=123):
    x = np.asarray(features_real, dtype=np.float64)
    y = np.asarray(features_fake, dtype=np.float64)
    m = min(len(x), len(y), subset_size)
    if m < 2:
        raise ValueError('Need at least 2 samples per set to compute KID.')

    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(num_subsets):
        x_idx = rng.choice(len(x), size=m, replace=False)
        y_idx = rng.choice(len(y), size=m, replace=False)
        x_sub = x[x_idx]
        y_sub = y[y_idx]

        k_xx = polynomial_kernel(x_sub, x_sub)
        k_yy = polynomial_kernel(y_sub, y_sub)
        k_xy = polynomial_kernel(x_sub, y_sub)

        mean_xx = (k_xx.sum() - np.trace(k_xx)) / (m * (m - 1))
        mean_yy = (k_yy.sum() - np.trace(k_yy)) / (m * (m - 1))
        mean_xy = k_xy.mean()
        estimates.append(mean_xx + mean_yy - 2.0 * mean_xy)

    return float(np.mean(estimates)), float(np.std(estimates))


## Exercise 6: extract features and score the fixed protocol

Goal:
- Apply the protocol you defined and compute the comparison scores.

Implementation idea:
1. Extract features for the real reference images, generated images, and noise images.
2. Split the real features into two halves so you can compute $\text{real} \leftrightarrow \text{real}$ as a sanity check.
3. Compute feature statistics for each group.
4. Compute `FID` for $\text{real} \leftrightarrow \text{real}$, $\text{real} \leftrightarrow \text{generated}$, and $\text{real} \leftrightarrow \text{noise}$.
5. Reuse the provided `KID` helper to compute the same three comparisons.
6. Print the results in a readable format.

Expected qualitative behavior:
- $\text{real} \leftrightarrow \text{real}$ should be the best case.
- $\text{real} \leftrightarrow \text{generated}$ should be worse than $\text{real} \leftrightarrow \text{real}$ but much better than random noise.
- If you change the protocol, the values may change even if the generator does not.


In [ ]:
# TODO START
# 1) Set feature_batch_size and reference_half from evaluation_config.
# 2) Extract features for real_eval_images, generated_images, and noise_images.
# 3) Split the real features into real_a and real_b.
# 4) Build gen_a and noise_a with the same number of samples as each real half.
# 5) Compute feature statistics for all required groups.
# 6) Compute the three FID comparisons.
# 7) Compute the three KID comparisons with the provided helper.
# 8) Print all scores in a readable format.
raise NotImplementedError('Implement feature extraction and fixed-protocol scoring.')
# TODO END


## RGB exploration with `diffusers`

Goal:
- After the MNIST exploration, repeat the workflow on an RGB generative model.
- Reuse the same interpolation and evaluation logic, but now with a matched RGB reference set.

Suggested workflow:
1. Start with `google/ddpm-cat-256`.
2. Generate a small RGB batch and inspect diversity, texture, and sample quality.
3. Test `lerp` and `slerp` between two RGB noise seeds while keeping the denoising budget fixed.
4. Load a real cat reference set from Oxford-IIIT Pet instead of reusing MNIST.
5. Reuse the same `FID` and `KID` helpers under a new RGB protocol.
6. If your machine can handle it, switch later to `google/ddpm-celebahq-256` and discuss what reference dataset must change.


In [ ]:
rgb_model_options = {
    'cat': 'google/ddpm-cat-256',
    'celebahq': 'google/ddpm-celebahq-256',
}

# TODO START
# Choose the RGB experiment configuration.
# Recommended first run: rgb_model_options['cat']
rgb_experiment = {
    'model_id': rgb_model_options['cat'],
    'n_samples': 6,
    'num_inference_steps': 50,
    'seed': 77,
    'interp_seed_a': 101,
    'interp_seed_b': 202,
    'interpolation_steps': 8,
}
# TODO END

print('RGB experiment config:', rgb_experiment)
rgb_pipeline = load_pretrained_ddpm(
    model_id=rgb_experiment['model_id'],
    cache_dir='hf-cache-rgb',
    device=device,
)
rgb_images = sample_random_ddpm(
    rgb_pipeline,
    n_samples=rgb_experiment['n_samples'],
    seed=rgb_experiment['seed'],
    num_inference_steps=rgb_experiment['num_inference_steps'],
)
show_image_grid(
    rgb_images,
    title=f"RGB diffusers samples: {rgb_experiment['model_id']}",
    nrow=min(3, rgb_experiment['n_samples']),
    figsize=(9, 6),
)


## RGB follow-up: test `lerp` and `slerp` on the cat DDPM

Goal:
- Apply the same interpolation protocol to the RGB `diffusers` cat model.
- Keep the denoising budget fixed so the comparison stays fair.

Hint:
- Build two single-sample noise tensors with different seeds.
- Reuse `build_interpolation_paths(...)` and `sample_ddpm_from_noises(...)`.


In [ ]:
# TODO START
# 1) Build t_values from rgb_experiment['interpolation_steps'] on device.
# 2) Create two RGB noises with seeds interp_seed_a and interp_seed_b, then squeeze the batch dimension.
# 3) Build lerp and slerp paths.
# 4) Decode both paths with sample_ddpm_from_noises using the same num_inference_steps.
# 5) Show a side-by-side comparison with show_interpolation_comparison.
raise NotImplementedError('Implement RGB DDPM interpolation with lerp and slerp.')
# TODO END


## RGB adaptation guide: evaluation on a cat reference set

To adapt the MNIST evaluation workflow to the RGB cat model, keep these changes explicit:
- Replace the MNIST reference set with real cat images from Oxford-IIIT Pet.
- Keep the generator domain and the reference domain matched: cat generator, cat reference images.
- Keep the feature extractor the same (`InceptionV3`), because the notebook already normalizes and resizes images consistently for feature extraction.
- Use smaller counts first because `256x256` RGB images are heavier than MNIST.
- Reuse the same comparison pairs: `real vs real`, `real vs generated`, and `real vs noise`.

Suggested first protocol for class:
1. Use `google/ddpm-cat-256`.
2. Load `64` real cat images from the Oxford-IIIT Pet `test` split.
3. Generate `64` fake cat images with the same number of denoising steps used in the RGB sampling section.
4. Keep the seeds visible in the config.
5. Only after the protocol is fixed, compute `FID` and `KID`.


In [ ]:
# TODO START
# Keep the RGB cat evaluation protocol explicit.
rgb_evaluation_config = {
    'reference_count': 64,
    'generated_count': 64,
    'feature_batch_size': 16,
    'subset_size': 16,
    'num_subsets': 10,
    'generation_seed': 404,
    'noise_seed': 505,
    'reference_split': 'test',
    'reference_image_size': 256,
    'num_inference_steps': rgb_experiment['num_inference_steps'],
}
# TODO END

if rgb_evaluation_config['reference_count'] % 2 != 0:
    raise ValueError('reference_count must be even so it can be split into two reference sets.')

rgb_feature_extractor = build_inception_feature_extractor(device=device)
rgb_real_images, _ = load_oxford_pet_cat_subset(
    limit=rgb_evaluation_config['reference_count'],
    split=rgb_evaluation_config['reference_split'],
    image_size=rgb_evaluation_config['reference_image_size'],
    data_root='data',
)
rgb_real_eval_images = rgb_real_images[:rgb_evaluation_config['reference_count']]
rgb_generated_images = sample_random_ddpm(
    rgb_pipeline,
    n_samples=rgb_evaluation_config['generated_count'],
    seed=rgb_evaluation_config['generation_seed'],
    num_inference_steps=rgb_evaluation_config['num_inference_steps'],
)
rgb_noise_generator = torch.Generator(device='cpu').manual_seed(rgb_evaluation_config['noise_seed'])
rgb_noise_images = torch.rand(rgb_real_eval_images.shape, generator=rgb_noise_generator)

print('RGB evaluation protocol:', rgb_evaluation_config)
show_image_grid(rgb_real_eval_images[:16], title='Real cat reference samples', nrow=4, figsize=(8, 8), cmap=None)
show_image_grid(rgb_generated_images[:16], title='Generated cat samples', nrow=4, figsize=(8, 8), cmap=None)
show_image_grid(rgb_noise_images[:16], title='Random RGB noise baseline', nrow=4, figsize=(8, 8), cmap=None)


## RGB follow-up: compute `FID` and `KID` for the cat DDPM

Reuse the same scoring logic from the MNIST section, but change only the protocol inputs:
- `rgb_real_eval_images` becomes the real reference set
- `rgb_generated_images` becomes the generated set
- `rgb_noise_images` becomes the noise baseline

Checklist:
1. Extract RGB features with `rgb_feature_extractor`.
2. Split the real cat features into two equal halves.
3. Compute `FID` for `real-real`, `real-generated`, and `real-noise`.
4. Compute `KID` for the same three pairs.
5. Compare the ordering with the MNIST case.


In [ ]:
# TODO START
# 1) Set rgb_feature_batch_size and rgb_reference_half from rgb_evaluation_config.
# 2) Extract features for rgb_real_eval_images, rgb_generated_images, and rgb_noise_images.
# 3) Split the real cat features into rgb_real_a and rgb_real_b.
# 4) Build rgb_gen_a and rgb_noise_a with the same number of samples as each real half.
# 5) Compute feature statistics for all required groups.
# 6) Compute RGB FID for real-real, real-generated, and real-noise.
# 7) Compute RGB KID for the same three comparisons.
# 8) Print the RGB scores in a readable format.
raise NotImplementedError('Implement RGB cat evaluation with the fixed protocol.')
# TODO END


## Questions for discussion

1. Which parts of the workflow transferred cleanly from MNIST to the RGB model?
2. Do the visual differences between `lerp` and `slerp` become easier or harder to read in the RGB model?
3. Which parts of the evaluation protocol had to be fixed before the `FID` numbers became interpretable?
4. Why is $\text{real} \leftrightarrow \text{real}$ an important sanity check?
5. If you switch from MNIST to `google/ddpm-cat-256` or `google/ddpm-celebahq-256`, what reference dataset and preprocessing would you need?
